In [1]:
try:
    import google.colab  # noqa: F401

    # specify the version of DataEval (==X.XX.X) for versions other than the latest
    %pip install -q dataeval dataeval-plots[plotly] maite-datasets
except Exception:
    pass

In [2]:
import dataeval_plots as dep
import numpy as np
import plotly.io as pio
import torch
from maite_datasets.image_classification import MilitaryVehicles
from torchvision.models import ResNet18_Weights, resnet18

from dataeval import Embeddings
from dataeval.config import set_device, set_max_processes, set_seed
from dataeval.core import cluster, coverage_adaptive
from dataeval.extractors import TorchExtractor
from dataeval.quality import Duplicates, Outliers

# Set default device for notebook
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_device(device)
set_seed(0, all_generators=True)  # fix the random seed for reproducible results

# Use plotly to render plots
dep.set_default_backend("plotly")

# Use the notebook renderer so JS is embedded
pio.renderers.default = "notebook"

# Set the maximum number of processes to use
set_max_processes(4)

In [3]:
# Load the training dataset
dataset = MilitaryVehicles(root="./data", image_set="train", download=True)

print(f"Images: {len(dataset)}")
print(f"Image shape: {dataset[0][0].shape}")

Images: 7823
Image shape: (3, 181, 278)


In [4]:
resnet = resnet18(weights=ResNet18_Weights.DEFAULT, progress=False)
extractor = TorchExtractor(resnet, transforms=ResNet18_Weights.DEFAULT.transforms(), layer_name="avgpool")
embeddings = np.asarray(Embeddings(dataset, extractor=extractor, batch_size=64))

print("embedding shape:", embeddings.shape)

embedding shape: (7823, 512)


In [5]:
output = cluster(embeddings)

In [6]:
duplicates = Duplicates().from_clusters(output).data()
groups = duplicates["item_indices"].to_list()

print(f"Duplicate groups: {len(groups)}")
print(f"Images in a group: {len({i for group in groups for i in group})}")

Duplicate groups: 67
Images in a group: 139


In [7]:
pairs = [group for group in groups if len(group) == 2][:3]
dep.plot(dataset, indices=[i for pair in pairs for i in pair], figsize=(10, 6), images_per_row=2)

In [8]:
index2label = dataset.index2label


def label_of(i: int) -> str:
    """Return the class name that the dataset gives to image `i`."""
    return index2label[int(np.argmax(dataset[i][1]))]


for pair in pairs:
    print([label_of(i) for i in pair])

conflicting = [group for group in groups if len({label_of(i) for i in group}) > 1]
print(f"Groups with more than one label: {len(conflicting)} of {len(groups)}")

['BM-30', 'T-64']
['BM-30', 'Tornado']
['BM-30', 'D-30']
Groups with more than one label: 67 of 67


In [9]:
# Use the Outliers.from_clusters() method
detector = Outliers()
result = detector.from_clusters(embeddings, output, cluster_threshold=3.0)

# Extract the outlier indices from the result
outliers = result.aggregate_by_item()["item_index"].to_list()

print(f"Number of outliers: {len(outliers)}")

Number of outliers: 12


In [10]:
print(f"Percent of outliers: {(100 * len(outliers) / len(dataset)):.2f}%")

Percent of outliers: 0.15%


In [11]:
# To use the flagged outliers, uncomment the line below
indices = outliers

# To use unflagged images, uncomment the line below
# indices = list(set(range(len(dataset))) - set(outliers))

In [12]:
# Plot up to 12 images from the dataset
dep.plot(
    dataset,
    indices=list(indices)[:12],
    figsize=(10, 8),
    images_per_row=4,
)

In [13]:
cov = coverage_adaptive(embeddings, 50, 0.01, force_unit_interval=True)

print(f"Number of uncovered images: {len(cov['uncovered_indices'])}")

Number of uncovered images: 78


In [14]:
# TEST ASSERTION CELL ###
from dataeval.scope import Coverage

labeled = Coverage(num_observations=50, percent=0.01).evaluate(
    np.zeros(len(embeddings), dtype=int), embeddings=embeddings
)
assert np.array_equal(np.sort(labeled.uncovered_indices), np.sort(cov["uncovered_indices"]))

In [15]:
uncovered_outliers = set(cov["uncovered_indices"].tolist()).intersection(set(outliers))

print(f"Number of outliers found as uncovered images: {len(uncovered_outliers)}")

Number of outliers found as uncovered images: 12


In [16]:
underrepresented = [i for i in cov["uncovered_indices"].tolist() if i not in set(outliers)]
dep.plot(dataset, indices=underrepresented[:6], figsize=(10, 6), images_per_row=3)

In [17]:
# TEST ASSERTION CELL ###
assert (len(dataset), len(dataset.index2label)) == (7823, 24)
assert embeddings.shape == (len(dataset), 512)
assert 0 < len(outliers) <= 0.05 * len(dataset)
assert len(cov["uncovered_indices"]) > 0
# The counts move a little with the hardware, so the checks below use bands the prose stays true in
assert len(pairs) == 3
assert all(len({label_of(i) for i in pair}) == 2 for pair in pairs)  # "filed under two different labels"
assert len(conflicting) >= 0.9 * len(groups)  # "nearly every other group is too"
assert len(set(outliers) & {2055, 5022, 7054, 5094, 4062, 7345, 7365, 7557}) >= 6  # junk and rocket launchers firing
assert len(uncovered_outliers) >= 0.8 * len(outliers)  # "most of the outliers are also uncovered"
assert len(cov["uncovered_indices"]) > len(outliers)  # "a larger group"
assert len(set(underrepresented[:6]) & {3267, 4928, 278, 5104, 3867}) >= 3  # models, close-ups, a ring, a flag